In [1]:
from config import CLASSES_TO_USE, ORGANISMS_TO_USE, PER_ORGANISM_PER_CLASS_ALLELES, RESULT_COLUMNS

#rewrite classes to use to only I class because it's only class I method
METHOD = 'NetMHCIIpan-4.0'
CLASSES_TO_USE = ['II']
ORGANISMS_TO_USE = ['Human']
ALLELES = [item  for organism_to_use in ORGANISMS_TO_USE 
                 for class_to_use in CLASSES_TO_USE 
                 for item in PER_ORGANISM_PER_CLASS_ALLELES[organism_to_use][class_to_use]]
ALLELES

['HLA_DRB1_01_01', 'HLA_DRB1_04_01', 'HLA_DRB1_04_05']

In [2]:
from mhctools import NetMHCIIpan4
from mhcnames import normalize_allele_name## ONLY FOR LINUX
import pandas as pd

In [3]:
DATA_PATH = "/home/denklewer/Data/"
TARGET_PATH_TO_PREDICTIONS = DATA_PATH + f'predictions/{METHOD}/'

# # Read datasets
# DECOY_KEY = 'x1'


# INPUT_DATA_PATH = DATA_PATH + 'input/'
# per_allele_val_df = {
#     allele_name :pd.read_csv(f"{INPUT_DATA_PATH}df_{allele_name}_{DECOY_KEY}_val_for_classification.csv", index_col=0) for allele_name in ALLELES
# }
# for allele_name in ALLELES:
#     per_allele_val_df[allele_name]['source_sequence_name'] =  'seq' + per_allele_val_df[allele_name].index.astype(str).values

In [4]:
from data_reading import read_genetech_data

per_allele_train_df, per_allele_val_df = read_genetech_data(DATA_PATH, CLASSES_TO_USE, ALLELES)


## Prediction code

In [5]:
per_allele_df = per_allele_val_df
TARGET_ALLELES = dict()

## HARDCODE allele renaming if needed
for allele_name in ALLELES:
    if allele_name == 'H2_d class II':
        target_allele_name = 'H2_IAd'
    elif allele_name == 'H2_b class II':
        target_allele_name = 'H2_IAb'
    else:
        target_allele_name = allele_name
    TARGET_ALLELES[allele_name] = (target_allele_name)


TARGET_ALLELES

{'HLA_DRB1_01_01': 'HLA_DRB1_01_01',
 'HLA_DRB1_04_01': 'HLA_DRB1_04_01',
 'HLA_DRB1_04_05': 'HLA_DRB1_04_05'}

In [6]:
def splitByLengthAndFilter(fasta_dict, min_length = 9):
    unique_lengths = list(set(map(len, list(fasta_dict.values()))))
    unique_lengths = list(filter(lambda x: x>=min_length, unique_lengths ))
    per_length_dict_of_dicts = {target_length :{k: v for k, v in fasta_dict.items() if len(v) == target_length} for target_length in unique_lengths}
    return per_length_dict_of_dicts

def predictSequences(dict_of_dicts, allele_name):
    alleles = [normalize_allele_name(TARGET_ALLELES[allele_name])]   # normalize allele name here
    predictor = NetMHCIIpan4(
            alleles=alleles,
            program_name='netMHCIIpan-4.0')
    print(f"Now we are going to predict binders for  dataset {allele_name} using alleles parameter {alleles}")
    binding_predictions = [predictor.predict_subsequences(fasta_dict, 
                                                          peptide_lengths=[length]) # peek any element and take length
                           for length, fasta_dict in dict_of_dicts.items()]
    lengths = [len(prediction) for prediction in binding_predictions] 
    print(f"total processed peptides after filtering -  {sum(lengths)}")
    return binding_predictions



def predict_with_netmhc(per_allele_df):
    # add ID as column
    for allele_name in ALLELES:
        per_allele_df[allele_name][RESULT_COLUMNS['id']] =  'seq' + per_allele_df[allele_name].index.astype(str).values
    # make fasta-style dicts
    per_allele_fasta = {
        allele_name : {
            ID : PEPTIDE 
            for ID, PEPTIDE in zip('seq' + per_allele_df[allele_name].index.astype(str).values, per_allele_df[allele_name]['peptide'].values) 
        } 
        for allele_name in ALLELES 
    }
    
    # split by length
    per_allele_per_length_dict = dict() 
    for allele_name in ALLELES:
        fasta_dict = per_allele_fasta[allele_name]
        per_length_dict_of_dicts = splitByLengthAndFilter(fasta_dict)
        print(f"allele {allele_name} fasta splited into {len(per_length_dict_of_dicts)} dicts")
        per_allele_per_length_dict[allele_name] = per_length_dict_of_dicts

    # predict
    per_allele_predictions = dict()
    for allele_name, per_length_dict in per_allele_per_length_dict.items():
        predictions = predictSequences(per_length_dict, allele_name)
        per_allele_predictions[allele_name] = predictions
    # join lengths together
    per_allele_total_predictions = dict()
    for allele_name, predictions in per_allele_predictions.items():
        list_of_df = [prediction.to_dataframe() for prediction in predictions]
        df_result = pd.concat(list_of_df, ignore_index=True)
        per_allele_total_predictions[allele_name] = df_result
        

    return per_allele_total_predictions

In [7]:
%%time
per_allele_total_predictions = predict_with_netmhc(per_allele_val_df)

allele HLA_DRB1_01_01 fasta splited into 22 dicts
allele HLA_DRB1_04_01 fasta splited into 22 dicts
allele HLA_DRB1_04_05 fasta splited into 22 dicts
Now we are going to predict binders for  dataset HLA_DRB1_01_01 using alleles parameter ['HLA-DRA1*01:01-DRB1*01:01']
total processed peptides after filtering -  22579
Now we are going to predict binders for  dataset HLA_DRB1_04_01 using alleles parameter ['HLA-DRA1*01:01-DRB1*04:01']
total processed peptides after filtering -  2777
Now we are going to predict binders for  dataset HLA_DRB1_04_05 using alleles parameter ['HLA-DRA1*01:01-DRB1*04:05']
total processed peptides after filtering -  1528
CPU times: user 1.19 s, sys: 398 ms, total: 1.59 s
Wall time: 7min 17s


#### assert we have right column names

In [8]:
per_allele_df = {key : df.rename(columns={"el":RESULT_COLUMNS['label']}) for  key,df in per_allele_df.items()}
for allele_name in per_allele_total_predictions.keys(): 
    df_result = per_allele_total_predictions[allele_name]
    per_allele_total_predictions[allele_name] = df_result.merge(per_allele_df[allele_name][[RESULT_COLUMNS['id'],RESULT_COLUMNS['label']]], on=['source_sequence_name'])


assert RESULT_COLUMNS['label'] in per_allele_total_predictions[ALLELES[0]].columns, f"label column must be in the result, this column is '{RESULT_COLUMNS['label']}'"
assert RESULT_COLUMNS['id'] in per_allele_total_predictions[ALLELES[0]].columns, f"id column must be in the result, this column is '{RESULT_COLUMNS['id']}'"
assert RESULT_COLUMNS['peptide'] in per_allele_total_predictions[ALLELES[0]].columns, f"peptide column must be in the result, this column is '{RESULT_COLUMNS['peptide']}'"

In [9]:
df = per_allele_total_predictions[ALLELES[0]]
df

,source_sequence_name,offset,peptide,allele,score,affinity,percentile_rank,prediction_method_name,length,binder,binder
0,seq63991,0,AAATSIAMS,HLA-DRA1*01:01-DRB1*01:01,0.000006,None,95.00,netMHCIIpan-4.0,9,0.0,0.0
1,seq66554,0,AALQEGLRR,HLA-DRA1*01:01-DRB1*01:01,0.000002,None,95.00,netMHCIIpan-4.0,9,0.0,0.0
2,seq69190,0,AALVALACA,HLA-DRA1*01:01-DRB1*01:01,0.000001,None,95.00,netMHCIIpan-4.0,9,0.0,0.0
3,seq65937,0,AAMSKDKAN,HLA-DRA1*01:01-DRB1*01:01,0.000005,None,95.00,netMHCIIpan-4.0,9,0.0,0.0
4,seq78444,0,AAPKSPVRE,HLA-DRA1*01:01-DRB1*01:01,0.000002,None,95.00,netMHCIIpan-4.0,9,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...
22574,seq79745,0,YVEDGTPCGPQMMCLEHRCLPVASFNFSTC,HLA-DRA1*01:01-DRB1*01:01,0.000392,None,91.79,netMHCIIpan-4.0,30,0.0,0.0
22575,seq63504,0,YVSTLASEGNYGESGVEAFVQISREAGGVC,HLA-DRA1*01:01-DRB1*01:01,0.004834,None,43.93,netMHCIIpan-4.0,30,0.0,0.0
22576,seq68990,0,YVWDWQRHEIVQTLSLKDGLIPLEIRFLHN,HLA-DRA1*01:01-DRB1*01:01,0.002048,None,61.41,netMHCIIpan-4.0,30,0.0,0.0
22577,seq65661,0,YYINHETQTTCWDHPKMTELYQSLADLNNV,HLA-DRA1*01:01-DRB1*01:01,0.001786,None,64.36,netMHCIIpan-4.0,30,0.0,0.0


#### write to files

In [10]:
# for decoy_key in DECOY_PREFIXES:
for allele_name, total_df in per_allele_total_predictions.items():
    total_df.to_csv(f"{TARGET_PATH_TO_PREDICTIONS}{METHOD}_predictions_{allele_name}")